In [3]:
import subprocess
import sys

# Install dotenv
print("Installing python-dotenv...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"])
print("✓ Installed!")

Installing python-dotenv...
✓ Installed!


In [4]:
import json
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Load dev data (we'll use this for testing LLM)
dev_data = []
with open("../scicite/dev.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        example = json.loads(line)
        dev_data.append({
            'text': example['string'],
            'label': example['label']
        })

dev_df = pd.DataFrame(dev_data)
y_dev = dev_df['label'].values
X_dev = dev_df['text'].values

print(f"Dev data loaded: {len(dev_data)} examples")
print(f"Classes: {np.unique(y_dev)}")

Dev data loaded: 916 examples
Classes: ['background' 'method' 'result']


In [5]:
import subprocess
import sys

# Install openai library
print("Installing openai...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
print("✓ Installed!")

Installing openai...
✓ Installed!


In [6]:
from openai import OpenAI
import os

# Load API key from .env file
api_key = os.getenv('LITELLM_API_KEY')

if api_key is None:
    print("ERROR: API key not found in .env file!")
    print("Make sure .env file has: LITELLM_API_KEY=your_key_here")
else:
    print(f"✓ API key loaded ...")
    
    # Create OpenAI client pointed at LiteLLM
    client = OpenAI(
        api_key=api_key,
        base_url="https://litellm.professor-x.de/v1",
    )
    
    print("✓ Client created")
    print("✓ Ready to use LiteLLM API")

✓ API key loaded ...
✓ Client created
✓ Ready to use LiteLLM API


In [10]:
# Send a test request to the LLM
print("Sending test request to LiteLLM...")

response = client.chat.completions.create(
    model="hosted_vllm/Qwen/Qwen3.6-35B-A3B-FP8",
    messages=[
        {"role": "user", "content": "What are the three citation intent classes in academic papers?"}
    ],
    temperature=0.2,
)

print("✓ Response received!")
print(f"\nLLM Answer:")
print(response.choices[0].message.content)

Sending test request to LiteLLM...
✓ Response received!

LLM Answer:


The three widely recognized **citation intent classes** in academic literature are:

1. **Background** (or *Contextual*):  
   Citations that provide foundational knowledge, establish the research landscape, or offer general context for the study. They answer "what is known" or "where does this work fit?"

2. **Comparison** (or *Contrast*):  
   Citations used to compare, contrast, or differentiate the current work from prior studies in terms of methods, results, assumptions, or conclusions. They highlight similarities, differences, or gaps.

3. **Support** (or *Justification/Validation*):  
   Citations that validate, justify, or lend credibility to the authors' claims, hypotheses, methodologies, or interpretations. They answer "why this approach/claim is reasonable or correct."

📖 **Origin & Usage**:  
This three-class taxonomy was formalized in computational citation analysis and natural language processing resea

In [11]:
# Now let's classify actual citations from SciCite

print("Classifying citations using LLM prompting...\n")

# Take first 10 examples from dev data
sample_size = 10
sample_texts = X_dev[:sample_size]
sample_labels = y_dev[:sample_size]

llm_predictions = []

for i, citation_text in enumerate(sample_texts):
    # Create the prompt
    prompt = f"""Classify the following academic citation into one of three categories:
    - background: Citation provides context or background information
    - method: Citation describes a method or approach being used
    - result: Citation compares or discusses results

Citation: "{citation_text}"

Answer with ONLY the category name (background, method, or result), nothing else."""

    # Send to LLM
    response = client.chat.completions.create(
        model="hosted_vllm/Qwen/Qwen3.6-35B-A3B-FP8",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
    )
    
    # Extract the prediction
    prediction = response.choices[0].message.content.strip().lower()
    llm_predictions.append(prediction)
    
    print(f"Example {i+1}:")
    print(f"  Text: {citation_text[:80]}...")
    print(f"  True: {sample_labels[i]}")
    print(f"  LLM:  {prediction}")
    print()

# Calculate accuracy on these 10
correct = sum(1 for pred, true in zip(llm_predictions, sample_labels) if pred == true)
accuracy = correct / len(sample_labels)
print(f"Accuracy on 10 samples: {accuracy:.1%}")

Classifying citations using LLM prompting...

Example 1:
  Text: These results are in contrast with the findings of Santos et al.(16), who report...
  True: result
  LLM:  result

Example 2:
  Text: …nest burrows in close proximity of one another appears to be well founded as pr...
  True: background
  LLM:  background

Example 3:
  Text: This is clearly in contrast to the results of earlier investigations ( Laprise &...
  True: result
  LLM:  result

Example 4:
  Text: …in a subset of alcoholics (Chen et al., 2004; McElroy et al., 2009; Mistlberger...
  True: background
  LLM:  background

Example 5:
  Text: This result is consistent with the conclusions of the aforementioned recent stud...
  True: result
  LLM:  result

Example 6:
  Text: Another examples of twisted bicrossproduct Hopf algebra are the null-plane quant...
  True: background
  LLM:  background

Example 7:
  Text: Our results confirm the other studies suggesting that antioxidants may have a pr...
  True: result
  LLM:  

In [12]:
# Show first 3 examples from dev data
print("=" * 70)
print("SCICITE DEV DATA - FIRST 3 EXAMPLES")
print("=" * 70)

for i in range(3):
    print(f"\n[Example {i+1}]")
    print(f"Citation text: {X_dev[i]}")
    print(f"True label: {y_dev[i]}")
    print()

SCICITE DEV DATA - FIRST 3 EXAMPLES

[Example 1]
Citation text: These results are in contrast with the findings of Santos et al.(16), who reported a significant association between low sedentary time and healthy CVF among Portuguese
True label: result


[Example 2]
Citation text: …nest burrows in close proximity of one another appears to be well founded as previously shown by several studies that measured distances between kin vs. non-kin nest burrows, including in long-term data sets (King 1989b; Viblanc et al. 2010; Arnaud, Dobson & Murie 2012; Dobson et al. 2012).
True label: background


[Example 3]
Citation text: This is clearly in contrast to the results of earlier investigations ( Laprise & Peltier 1989a , Pierre - humbert & Wyman 1985 , Clark & Peltier 1977 ) , where it was found that the criteria for static and dynamic instabilities are simultaneously satisfied .
True label: result



In [14]:
print("Classifying ALL dev data with LLM...")
print(f"Total: {len(X_dev)} citations\n")

llm_predictions = []

for i, citation_text in enumerate(X_dev):
    # Create the prompt
    prompt = f"""Classify this citation into one of three categories:
- background: Context or background information
- method: Describes a method or approach
- result: Compares or discusses results

Citation: "{citation_text}"

Answer with ONLY the category (background, method, or result)."""

    # Send to LLM
    response = client.chat.completions.create(
        model="hosted_vllm/Qwen/Qwen3.6-35B-A3B-FP8",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
    )
    
    # Extract prediction
    prediction = response.choices[0].message.content.strip().lower()
    llm_predictions.append(prediction)
    
    # Show progress every 100
    if (i + 1) % 100 == 0:
        print(f"Processed: {i+1}/{len(X_dev)}")

print("\n✓ Classification complete!")

# Convert to array
llm_predictions = np.array(llm_predictions)

# Calculate metrics
dev_accuracy = accuracy_score(y_dev, llm_predictions)
dev_f1 = f1_score(y_dev, llm_predictions, average='macro')

print(f"\nDev Results (LLM Prompting):")
print(f"  Accuracy: {dev_accuracy:.4f}")
print(f"  Macro F1: {dev_f1:.4f}")

Classifying ALL dev data with LLM...
Total: 916 citations

Processed: 100/916
Processed: 200/916
Processed: 300/916
Processed: 400/916
Processed: 500/916
Processed: 600/916
Processed: 700/916
Processed: 800/916
Processed: 900/916

✓ Classification complete!

Dev Results (LLM Prompting):
  Accuracy: 0.6747
  Macro F1: 0.6576


In [7]:
# Few-shot prompting with examples
print("Trying few-shot prompting with examples...")
print(f"Total: {len(X_dev)} citations\n")

# Define examples
few_shot_examples = """Examples:
- "We use the BERT model from Devlin et al." → method
- "Previous studies showed that..." → background
- "Unlike Smith et al., our results are..." → result"""

llm_predictions_fewshot = []

for i, citation_text in enumerate(X_dev):
    prompt = f"""{few_shot_examples}

Now classify this citation:
Citation: "{citation_text}"

Answer ONLY with: background, method, or result"""

    response = client.chat.completions.create(
        model="hosted_vllm/Qwen/Qwen3.6-35B-A3B-FP8",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
    )
    
    prediction = response.choices[0].message.content.strip().lower()
    llm_predictions_fewshot.append(prediction)
    
    if (i + 1) % 100 == 0:
        print(f"Processed: {i+1}/{len(X_dev)}")

print("\n✓ Few-shot classification complete!")

# Calculate metrics
dev_accuracy_fewshot = accuracy_score(y_dev, llm_predictions_fewshot)
dev_f1_fewshot = f1_score(y_dev, llm_predictions_fewshot, average='macro')

print(f"\nDev Results (LLM Few-shot):")
print(f"  Accuracy: {dev_accuracy_fewshot:.4f}")
print(f"  Macro F1: {dev_f1_fewshot:.4f}")

Trying few-shot prompting with examples...
Total: 916 citations

Processed: 100/916
Processed: 200/916
Processed: 300/916
Processed: 400/916
Processed: 500/916
Processed: 600/916
Processed: 700/916
Processed: 800/916
Processed: 900/916

✓ Few-shot classification complete!

Dev Results (LLM Few-shot):
  Accuracy: 0.7325
  Macro F1: 0.6959


In [8]:
import json

# Store all results
all_results = {
    'TF-IDF + Naive Bayes': {
        'accuracy': 0.7445,
        'f1': 0.6641,
        'approach': 'Simple baseline'
    },
    'SciBERT + Logistic Regression': {
        'accuracy': 0.8242,
        'f1': 0.7973,
        'approach': 'Fine-tuned encoder'
    },
    'LLM Zero-shot': {
        'accuracy': 0.6747,
        'f1': 0.6576,
        'approach': 'LLM prompting (basic)'
    },
    'LLM Few-shot': {
        'accuracy': 0.7325,
        'f1': 0.6959,
        'approach': 'LLM prompting (with examples)'
    }
}

# Print comparison
print("=" * 70)
print("JUNE 1 MILESTONE - CLASSIFIER COMPARISON")
print("=" * 70)
print("\nTaxonomy: SciCite (3 classes: background, method, result)")
print("Dataset: 916 dev examples\n")

# Sort by accuracy
sorted_results = sorted(all_results.items(), key=lambda x: x[1]['accuracy'], reverse=True)

for rank, (model_name, metrics) in enumerate(sorted_results, 1):
    print(f"{rank}. {model_name}")
    print(f"   Accuracy: {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
    print(f"   Macro F1: {metrics['f1']:.4f}")
    print(f"   Approach: {metrics['approach']}")
    print()

# Find best
best_model = sorted_results[0][0]
best_accuracy = sorted_results[0][1]['accuracy']
print(f"✓ BEST MODEL: {best_model} ({best_accuracy*100:.2f}% accuracy)")

# Save to file
with open('../june1_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("\n✓ Results saved to june1_results.json")

JUNE 1 MILESTONE - CLASSIFIER COMPARISON

Taxonomy: SciCite (3 classes: background, method, result)
Dataset: 916 dev examples

1. SciBERT + Logistic Regression
   Accuracy: 0.8242 (82.42%)
   Macro F1: 0.7973
   Approach: Fine-tuned encoder

2. TF-IDF + Naive Bayes
   Accuracy: 0.7445 (74.45%)
   Macro F1: 0.6641
   Approach: Simple baseline

3. LLM Few-shot
   Accuracy: 0.7325 (73.25%)
   Macro F1: 0.6959
   Approach: LLM prompting (with examples)

4. LLM Zero-shot
   Accuracy: 0.6747 (67.47%)
   Macro F1: 0.6576
   Approach: LLM prompting (basic)

✓ BEST MODEL: SciBERT + Logistic Regression (82.42% accuracy)

✓ Results saved to june1_results.json
